In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score
import re
import warnings
warnings.filterwarnings('ignore')

In [2]:
train = pd.read_csv('train.csv')
valid = pd.read_csv('valid.csv')
test  = pd.read_csv('test.csv')

y_train_hazard  = train['hazard-category']
y_train_product = train['product-category']
y_valid_hazard  = valid['hazard-category']
y_valid_product = valid['product-category']

print(f'Train: {len(train)} δείγματα')
print(f'Valid: {len(valid)} δείγματα')
print(f'Test:  {len(test)} δείγματα')

Train: 5082 δείγματα
Valid: 565 δείγματα
Test:  997 δείγματα


In [3]:
def preprocess_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def official_st1_score(y_true_hazard, y_pred_hazard,
                       y_true_product, y_pred_product,
                       verbose=True):
    f1_hazard = f1_score(y_true_hazard, y_pred_hazard,
                         average='macro', zero_division=0)
    
    y_true_hazard  = np.array(y_true_hazard)
    y_pred_hazard  = np.array(y_pred_hazard)
    y_true_product = np.array(y_true_product)
    y_pred_product = np.array(y_pred_product)
    
    correct_hazard_mask = (y_true_hazard == y_pred_hazard)
    n_correct = correct_hazard_mask.sum()
    
    if n_correct == 0:
        f1_product = 0.0
    else:
        f1_product = f1_score(
            y_true_product[correct_hazard_mask],
            y_pred_product[correct_hazard_mask],
            average='macro', zero_division=0
        )
    
    score = (f1_hazard + f1_product) / 2
    
    if verbose:
        print(f'macro-F1 Hazard:                    {f1_hazard:.4f}')
        print(f'Correct hazard predictions:           {n_correct}/{len(y_true_hazard)} ({100*n_correct/len(y_true_hazard):.1f}%)')
        print(f'macro-F1 Product (given correct h): {f1_product:.4f}')
        print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
        print(f'OFFICIAL ST1 SCORE:                 {score:.4f}')
    
    return score


In [4]:
# Μοντέλο Α: title + text[:550] — best model
train['combined'] = train['title'].fillna('') + ' ' + train['text'].fillna('').str[:550]
valid['combined'] = valid['title'].fillna('') + ' ' + valid['text'].fillna('').str[:550]
test['combined']  = test['title'].fillna('')  + ' ' + test['text'].fillna('').str[:550]

train['combined'] = train['combined'].apply(preprocess_text)
valid['combined'] = valid['combined'].apply(preprocess_text)
test['combined']  = test['combined'].apply(preprocess_text)

tfidf_A = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    stop_words='english'
)

X_train_A = tfidf_A.fit_transform(train['combined'])
X_valid_A = tfidf_A.transform(valid['combined'])
X_test_A  = tfidf_A.transform(test['combined'])

clf_A_hazard = LinearSVC(C=0.5, class_weight='balanced', max_iter=2000, random_state=42)
clf_A_hazard.fit(X_train_A, y_train_hazard)

clf_A_product = LinearSVC(C=0.5, class_weight='balanced', max_iter=2000, random_state=42)
clf_A_product.fit(X_train_A, y_train_product)

print('=== Μοντέλο Α (title + text[:550]) ===\n')
score_A = official_st1_score(
    valid['hazard-category'], clf_A_hazard.predict(X_valid_A),
    valid['product-category'], clf_A_product.predict(X_valid_A)
)

=== Μοντέλο Α (title + text[:550]) ===

macro-F1 Hazard:                    0.8040
Correct hazard predictions:           526/565 (93.1%)
macro-F1 Product (given correct h): 0.6798
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OFFICIAL ST1 SCORE:                 0.7419


In [5]:
# Μοντέλο Β: title only
train['title_clean'] = train['title'].apply(preprocess_text)
valid['title_clean'] = valid['title'].apply(preprocess_text)
test['title_clean']  = test['title'].apply(preprocess_text)

tfidf_B = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    stop_words='english'
)

X_train_B = tfidf_B.fit_transform(train['title_clean'])
X_valid_B = tfidf_B.transform(valid['title_clean'])
X_test_B  = tfidf_B.transform(test['title_clean'])

clf_B_hazard = LinearSVC(C=0.5, class_weight='balanced', max_iter=2000, random_state=42)
clf_B_hazard.fit(X_train_B, y_train_hazard)

clf_B_product = LinearSVC(C=0.5, class_weight='balanced', max_iter=2000, random_state=42)
clf_B_product.fit(X_train_B, y_train_product)

print('=== Μοντέλο Β (title only) ===\n')
score_B = official_st1_score(
    valid['hazard-category'], clf_B_hazard.predict(X_valid_B),
    valid['product-category'], clf_B_product.predict(X_valid_B)
)

=== Μοντέλο Β (title only) ===

macro-F1 Hazard:                    0.7061
Correct hazard predictions:           476/565 (84.2%)
macro-F1 Product (given correct h): 0.6717
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OFFICIAL ST1 SCORE:                 0.6889


In [6]:
from scipy.stats import mode

def ensemble_predict(preds_list):
    """
    Majority voting: για κάθε δείγμα επιλέγει την κλάση
    που προβλέπουν οι περισσότεροι classifiers.
    Αν ισοψηφία, επιλέγει το πρώτο μοντέλο (το καλύτερο).
    """
    preds_array = np.array(preds_list)  # shape: (n_models, n_samples)
    final_preds = []
    
    for i in range(preds_array.shape[1]):
        votes = preds_array[:, i]
        # Βρίσκουμε την πιο συχνή κλάση
        values, counts = np.unique(votes, return_counts=True)
        final_preds.append(values[np.argmax(counts)])
    
    return np.array(final_preds)


# Predictions από κάθε μοντέλο
pred_A_hazard  = clf_A_hazard.predict(X_valid_A)
pred_A_product = clf_A_product.predict(X_valid_A)

pred_B_hazard  = clf_B_hazard.predict(X_valid_B)
pred_B_product = clf_B_product.predict(X_valid_B)

# Ensemble predictions
ensemble_hazard  = ensemble_predict([pred_A_hazard, pred_B_hazard])
ensemble_product = ensemble_predict([pred_A_product, pred_B_product])

print('=== ΑΞΙΟΛΟΓΗΣΗ ENSEMBLE ===\n')
score_ensemble = official_st1_score(
    valid['hazard-category'], ensemble_hazard,
    valid['product-category'], ensemble_product
)

print(f'\n=== ΣΥΓΚΡΙΣΗ ===')
print(f'Μοντέλο Α (title+text[:550]): 0.7419')
print(f'Μοντέλο Β (title only):       0.6889')
print(f'Ensemble (A+B):               {score_ensemble:.4f}')

=== ΑΞΙΟΛΟΓΗΣΗ ENSEMBLE ===

macro-F1 Hazard:                    0.7534
Correct hazard predictions:           502/565 (88.8%)
macro-F1 Product (given correct h): 0.6728
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OFFICIAL ST1 SCORE:                 0.7131

=== ΣΥΓΚΡΙΣΗ ===
Μοντέλο Α (title+text[:550]): 0.7419
Μοντέλο Β (title only):       0.6889
Ensemble (A+B):               0.7131


## Cell Ensemble-1: Majority Voting Ensemble

Συνδυασμός Μοντέλου Α (title+text[:550]) και
Μοντέλου Β (title only) με majority voting.

**Αποτελέσματα:**
| Μοντέλο | Score |
|---|---|
| Μοντέλο Α (title+text[:550]) | 0.7419 |
| Μοντέλο Β (title only) | 0.6889 |
| Ensemble A+B (majority voting) | 0.7131 |

**Συμπέρασμα:** Το ensemble απέδωσε χειρότερα από το
Μοντέλο Α μόνο. Όταν τα μοντέλα διαφωνούν, το
χειρότερο μοντέλο "τραβάει προς τα κάτω" το αποτέλεσμα.
**Επόμενη δοκιμή:** Weighted voting με μεγαλύτερο βάρος
στο Μοντέλο Α.

In [7]:
def weighted_ensemble_predict(preds_list, weights):
    """
    Weighted voting: κάθε μοντέλο ψηφίζει με διαφορετικό βάρος.
    
    Args:
        preds_list: λίστα από predictions κάθε μοντέλου
        weights: βάρος κάθε μοντέλου (πρέπει να αθροίζουν σε 1)
    """
    preds_array = np.array(preds_list)
    final_preds = []
    
    for i in range(preds_array.shape[1]):
        votes = preds_array[:, i]
        # Για κάθε μοναδική κλάση, αθροίζουμε τα βάρη
        classes = np.unique(votes)
        class_weights = {}
        for cls in classes:
            class_weights[cls] = sum(weights[j] for j in range(len(votes)) if votes[j] == cls)
        # Επιλέγουμε την κλάση με το μεγαλύτερο συνολικό βάρος
        final_preds.append(max(class_weights, key=class_weights.get))
    
    return np.array(final_preds)


# Δοκιμή διαφορετικών βαρών
weight_combinations = [
    [0.9, 0.1],
    [0.8, 0.2],
    [0.7, 0.3],
    [0.6, 0.4],
]

print('=== WEIGHTED ENSEMBLE ===\n')
for weights in weight_combinations:
    ens_h = weighted_ensemble_predict([pred_A_hazard, pred_B_hazard], weights)
    ens_p = weighted_ensemble_predict([pred_A_product, pred_B_product], weights)
    
    score = official_st1_score(
        valid['hazard-category'], ens_h,
        valid['product-category'], ens_p,
        verbose=False
    )
    print(f'Βάρη A={weights[0]}, B={weights[1]} → Score: {score:.4f}')

print(f'\nΜοντέλο Α μόνο: 0.7419')

=== WEIGHTED ENSEMBLE ===

Βάρη A=0.9, B=0.1 → Score: 0.7419
Βάρη A=0.8, B=0.2 → Score: 0.7419
Βάρη A=0.7, B=0.3 → Score: 0.7419
Βάρη A=0.6, B=0.4 → Score: 0.7419

Μοντέλο Α μόνο: 0.7419


## Cell Ensemble-2: Weighted Voting Ensemble

Δοκιμή weighted voting με διαφορετικά βάρη.

| Βάρη (A, B) | Score |
|---|---|
| (0.9, 0.1) | 0.7419 |
| (0.8, 0.2) | 0.7419 |
| (0.7, 0.3) | 0.7419 |
| (0.6, 0.4) | 0.7419 |

**Συμπέρασμα:** Το weighted ensemble δεν βελτιώνει το
Μοντέλο Α. Τα δύο μοντέλα είναι πολύ παρόμοια (TF-IDF+SVM)
οπότε δεν αλληλοσυμπληρώνονται. Για αποτελεσματικό
ensemble χρειαζόμαστε διαφορετικούς τύπους μοντέλων
π.χ. TF-IDF+SVM + fine-tuned BERT.